# 03 | Análise exploratória de vendas

As análises abaixo respondem perguntas de negócio; os gráficos servem para comparar resultados, não são um fim em si. Receita considera desconto. Ticket médio é calculado por pedido. Margem é uma estimativa baseada no custo cadastrado do produto.

In [ ]:
from pathlib import Path; import sqlite3; import sys; import pandas as pd; import matplotlib.pyplot as plt
root = Path.cwd().resolve(); database = root / 'data' / 'processed' / 'retail_sales.db'; assert database.exists(), 'Execute primeiro o notebook 02'; connection = sqlite3.connect(database)
pd.options.display.float_format = lambda value: f'R$ {value:,.2f}'; plt.rcParams.update({'figure.figsize': (10, 5), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})
def query(sql): return pd.read_sql_query(sql, connection)

## 1. Qual foi o faturamento total e o ticket médio?

In [ ]:
kpis = query('SELECT ROUND(SUM(net_revenue), 2) AS faturamento, COUNT(DISTINCT order_id) AS pedidos, ROUND(SUM(net_revenue) / COUNT(DISTINCT order_id), 2) AS ticket_medio FROM sales'); display(kpis); print(f"Resposta: {int(kpis.pedidos.iloc[0]):,} pedidos geraram {kpis.faturamento.iloc[0]:,.2f} em receita líquida, com ticket médio de {kpis.ticket_medio.iloc[0]:,.2f}.")

## 2 e 7. Como o faturamento evoluiu e quais meses cresceram?

In [ ]:
monthly = query('SELECT substr(sale_date, 1, 7) AS mes, SUM(net_revenue) AS faturamento FROM sales GROUP BY substr(sale_date, 1, 7) ORDER BY mes'); monthly['crescimento_pct'] = monthly.faturamento.pct_change(fill_method=None) * 100; display(monthly.tail(12).round(2)); monthly.plot(x='mes', y='faturamento', marker='o', legend=False, title='Faturamento líquido mensal'); plt.ylabel('Receita (R$)'); plt.xlabel('Mês'); plt.xticks(rotation=45); plt.tight_layout(); plt.show(); best_growth = monthly.dropna(subset=['crescimento_pct']).nlargest(1, 'crescimento_pct').iloc[0]; print(f"Maior crescimento mensal: {best_growth.mes} ({best_growth.crescimento_pct:.1f}%).")

## 3. Quais produtos lideram em faturamento e unidades vendidas?

In [ ]:
products = query('SELECT p.product_name, p.category, SUM(s.quantity) AS unidades, SUM(s.net_revenue) AS faturamento FROM sales s JOIN products p ON p.product_id = s.product_id GROUP BY p.product_id, p.product_name, p.category ORDER BY faturamento DESC LIMIT 10'); display(products); products.sort_values('faturamento').plot.barh(x='product_name', y='faturamento', legend=False, title='Top 10 produtos por faturamento'); plt.xlabel('Receita líquida (R$)'); plt.ylabel('Produto'); plt.tight_layout(); plt.show(); print(f"Produto líder em faturamento: {products.iloc[0].product_name} ({products.iloc[0].faturamento:,.2f}).")

## 4. Qual região concentra o maior faturamento?

In [ ]:
regions = query('SELECT r.region_name, COUNT(DISTINCT s.order_id) AS pedidos, SUM(s.net_revenue) AS faturamento FROM sales s JOIN regions r ON r.region_id = s.region_id GROUP BY r.region_id, r.region_name ORDER BY faturamento DESC'); display(regions); regions.plot.bar(x='region_name', y='faturamento', legend=False, title='Faturamento por região'); plt.xlabel('Região'); plt.ylabel('Receita líquida (R$)'); plt.xticks(rotation=0); plt.tight_layout(); plt.show(); print(f"Região líder: {regions.iloc[0].region_name}, com {regions.iloc[0].faturamento:,.2f}.")

## 5. Quais vendedores geram maior volume de vendas?

In [ ]:
sellers = query('SELECT v.seller_name, COUNT(DISTINCT s.order_id) AS pedidos, SUM(s.quantity) AS unidades, SUM(s.net_revenue) AS faturamento FROM sales s JOIN sellers v ON v.seller_id = s.seller_id GROUP BY v.seller_id, v.seller_name ORDER BY faturamento DESC'); display(sellers.head(10)); print(f"Vendedor líder: {sellers.iloc[0].seller_name}, com {int(sellers.iloc[0].pedidos)} pedidos e {sellers.iloc[0].faturamento:,.2f} em receita.")

## 6. Quantos clientes são recorrentes e quanto representam?

In [ ]:
recurring = query('SELECT customer_id, COUNT(DISTINCT order_id) AS pedidos, SUM(net_revenue) AS faturamento FROM sales GROUP BY customer_id HAVING COUNT(DISTINCT order_id) >= 2 ORDER BY faturamento DESC'); total_customers = query('SELECT COUNT(DISTINCT customer_id) AS total FROM sales').total.iloc[0]; recurring_revenue = recurring.faturamento.sum(); total_revenue = kpis.faturamento.iloc[0]; print(f'Clientes recorrentes: {len(recurring):,} de {total_customers:,} ({len(recurring) / total_customers:.1%}).'); print(f'Receita desses clientes: R$ {recurring_revenue:,.2f} ({recurring_revenue / total_revenue:.1%} do total).'); display(recurring.head(10))

## 7. Qual é a margem estimada por categoria?

In [ ]:
margins = query('SELECT p.category, SUM(s.net_revenue) AS faturamento, SUM(s.estimated_margin) AS margem_estimada, 100.0 * SUM(s.estimated_margin) / SUM(s.net_revenue) AS margem_pct FROM sales s JOIN products p ON p.product_id = s.product_id GROUP BY p.category ORDER BY margem_estimada DESC'); display(margins.round(2)); margins.plot.bar(x='category', y='margem_pct', legend=False, title='Margem estimada por categoria'); plt.xlabel('Categoria'); plt.ylabel('Margem estimada (%)'); plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show(); print(f"Maior contribuição estimada de margem: {margins.iloc[0].category} ({margins.iloc[0].margem_pct:.1f}% de margem).")

## 8. Quais categorias são mais relevantes para o negócio?

In [ ]:
categories = query('SELECT p.category, SUM(s.quantity) AS unidades, SUM(s.net_revenue) AS faturamento FROM sales s JOIN products p ON p.product_id = s.product_id GROUP BY p.category ORDER BY faturamento DESC'); categories['participacao_pct'] = 100 * categories.faturamento / categories.faturamento.sum(); display(categories.round(2)); leader = categories.iloc[0]; print(f"Categoria líder: {leader.category}, responsável por {leader.participacao_pct:.1f}% da receita. Compare participação e margem antes de priorizar o mix.")

## Leitura para decisão

Use os rankings para identificar concentração e orientar investigação comercial, não como prova de causalidade. A margem depende de custos simulados; antes de decisões financeiras, substitua-os por custos reais e valide a política de descontos. O crescimento compara meses consecutivos e pode refletir sazonalidade.